# STAIR-Enhanced v4: STAIR-NLGCL (Neighborhood-Enriched Graph Contrastive Learning)

## 📌 Giới thiệu Kiến trúc v4 (STAIR-NLGCL)
Phiên bản **STAIR-NLGCL v4** tích hợp cơ chế **Neighborhood-enriched Graph Contrastive Learning (NLGCL)** vào STAIR baseline, tuân thủ 3 nguyên tắc thiết kế cốt lõi:

1. **Không phân chia cứng (NO Hard Split) không gian 64-D:** Giữ trọn vẹn đường cong suy giảm phổ β₃(d) liên tục, áp dụng L2-normalization toàn chiều.
2. **Zero-cost Augmentation:** Trích xuất trực tiếp các biểu diễn tầng trung gian {H⁰, H¹, H², H³} từ FSC đã tính sẵn — không tốn thêm bất kỳ FLOPs nào.
3. **In-batch Heterogeneous InfoNCE (VRAM-safe):** Đối chiếu chéo thực thể User↔Item giữa các tầng, dùng LogSumExp chống tràn số, chi phí chỉ ~30 MB.

### ⚙️ Siêu tham số Mới (v4)
| Tham số | CLI Flag | Mặc định | Ý nghĩa |
|:---|:---|:---:|:---|
| λ_nlgcl | `--lambda-nlgcl` | `1e-2` (0.01) | Trọng số NLGCL loss |
| τ | `--nlgcl-tau` | `0.2` | Nhiệt độ InfoNCE |
| G | `--nlgcl-G` | `1` | Số khoảng cách tầng đối chiếu |
| α | `--nlgcl-alpha` | `0.5` | Cân bằng User CL / Item CL |

### 📋 Quy trình Thực nghiệm
1. Clone repo & cài dependencies
2. Chuẩn bị dữ liệu từ Kaggle Input
3. Kiểm tra modules & configs
4. Huấn luyện Baby → Sports → Electronics
5. Tổng hợp kết quả & Ablation Study
6. Vẽ Learning Curves


## Cell 1 — Thiết lập Môi trường & Cài đặt STAIR-Enhanced


In [ ]:
# Cell 1: Môi trường & Cài đặt Dependencies
import os, shutil, subprocess, sys

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
os.chdir('/kaggle/working')

# 1. Luôn clone mới nhất từ repository
if os.path.exists(STAIR_DIR):
    print('Làm sạch thư mục cũ để clone mới nhất...')
    shutil.rmtree(STAIR_DIR, ignore_errors=True)

print('Cloning STAIR-Enhanced repository (branch main)...')
subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/ThanhChuong12/STAIR-Enhanced.git', STAIR_DIR
], check=True)

for p in [STAIR_DIR, '/kaggle/working']:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(STAIR_DIR)

# 2. Cài đặt đầy đủ các thư viện cần thiết (freerec, torchdata, torch-geometric, prettytable)
print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'torchdata==0.7.1'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'freerec==0.8.5', 'nvidia-ml-py', 'prettytable', 'matplotlib', 'pyyaml'], check=True)

import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_TAG  = 'cu' + torch.version.cuda.replace('.','') if torch.cuda.is_available() else 'cpu'
print(f'Installing torch-geometric for torch={TORCH_VER}+{CUDA_TAG}...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric', '-f', f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html'], check=False)

import freerec
print('=' * 60)
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB')
print(f'FreeRec : {freerec.__version__}')
print('=' * 60)

# 3. Kiểm tra file training script v4
v4_script = os.path.join(STAIR_DIR, 'main_stair_nlgcl_v4.py')
nlgcl_module = os.path.join(STAIR_DIR, 'models', 'stair_nlgcl.py')
assert os.path.exists(v4_script), f'Không tìm thấy {v4_script}'
assert os.path.exists(nlgcl_module), f'Không tìm thấy {nlgcl_module}'
print(f'[OK] main_stair_nlgcl_v4.py: {os.path.getsize(v4_script)} bytes')
print(f'[OK] models/stair_nlgcl.py: {os.path.getsize(nlgcl_module)} bytes')
print('[OK] Environment setup complete!')

## Cell 2 — Chuẩn bị Dữ liệu từ Kaggle Input (Tự động quét linh hoạt)


In [ ]:
# Cell 2: Chuẩn bị dữ liệu từ Kaggle Input sang /kaggle/data & STAIR-Enhanced/data
import os, shutil, glob

DATA_ROOTS = ['/kaggle/data', '/kaggle/working/STAIR-Enhanced/data']
for root in DATA_ROOTS:
    os.makedirs(root, exist_ok=True)

print('Các thư mục có trong /kaggle/input:')
if os.path.exists('/kaggle/input'):
    for item in os.listdir('/kaggle/input'):
        print(f'  - /kaggle/input/{item}')

REQUIRED_EXTENSIONS = ('.npy', '.pkl', '.txt', '.inter', '.item', '.pt')

def copy_dataset(keywords, full_name):
    copied_files = 0
    for root_dir, _, files in os.walk('/kaggle/input'):
        if any(kw.lower() in root_dir.lower() for kw in keywords):
            for f in files:
                if f.endswith(REQUIRED_EXTENSIONS):
                    src_path = os.path.join(root_dir, f)
                    for target_root in DATA_ROOTS:
                        dest_dir = os.path.join(target_root, full_name)
                        os.makedirs(dest_dir, exist_ok=True)
                        shutil.copy(src_path, os.path.join(dest_dir, f))
                    copied_files += 1
    
    check_dir = os.path.join(DATA_ROOTS[0], full_name)
    num_present = len(os.listdir(check_dir)) if os.path.exists(check_dir) else 0
    if num_present > 0:
        print(f'[OK] {full_name}: {copied_files} files copied (Tổng hiện có: {num_present} files).')
    else:
        print(f'[WARN] {full_name}: Không tìm thấy file trong /kaggle/input với keywords={keywords}')

# Sao chép theo từ khóa linh hoạt (khớp cả baby, sports, electronics ở bất kỳ thư mục con nào)
copy_dataset(['baby',        'amazon2014baby'],        'Amazon2014Baby_550_MMRec')
copy_dataset(['sports',      'amazon2014sports'],      'Amazon2014Sports_550_MMRec')
copy_dataset(['electronics', 'amazon2014electronics'], 'Amazon2014Electronics_550_MMRec')

print(f'\nDữ liệu sẵn sàng tại: {DATA_ROOTS[0]}')

## Cell 3 — Kiểm tra NLGCL Module & Configs trước khi huấn luyện


In [ ]:
# Cell 3: Kiểm tra NLGCL Module & STAIR-NLGCL v4 configs
import sys, os, yaml, torch
import torch.nn.functional as F

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
for p in [STAIR_DIR, '/kaggle/working']:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(STAIR_DIR)

# 1. Test NLGCL_Module (standalone, không cần freerec)
from models.stair_nlgcl import NLGCL_Module

N_u, N_i, D, B = 100, 50, 64, 16
module = NLGCL_Module(n_users=N_u, n_items=N_i, G=1, tau=0.2, alpha=0.5)
layer_embeds = [torch.randn(N_u + N_i, D, requires_grad=True) for _ in range(4)]
users = torch.randint(0, N_u, (B,))
pos_items = torch.randint(0, N_i, (B,))

loss = module(layer_embeds, users, pos_items)
loss.backward()

print(f'[1/3] NLGCL_Module Test:')
print(f'  Loss (G=1, tau=0.2): {loss.item():.4f}')
print(f'  Grad flow L0: {layer_embeds[0].grad.norm().item():.6f}')
print(f'  Grad flow L1: {layer_embeds[1].grad.norm().item():.6f}')
assert not torch.isnan(loss), 'Loss is NaN!'
print(f'  ✅ NLGCL_Module OK')

# 2. Numeric stability
layer_large = [torch.randn(N_u + N_i, D) * 100 for _ in range(4)]
for t in layer_large: t.requires_grad_(True)
loss_large = module(layer_large, users, pos_items)
assert not torch.isnan(loss_large) and not torch.isinf(loss_large)
print(f'\n[2/3] Numeric Stability: Loss={loss_large.item():.4f} ✅ LogSumExp stable')

# 3. YAML configs
configs = [
    f'{STAIR_DIR}/configs/Amazon2014Baby_550_MMRec.yaml',
    f'{STAIR_DIR}/configs/Amazon2014Sports_550_MMRec.yaml',
    f'{STAIR_DIR}/configs/Amazon2014Electronics_550_MMRec.yaml',
]
print(f'\n[3/3] Kiểm tra YAML configs:')
for cfg_path in configs:
    if os.path.exists(cfg_path):
        with open(cfg_path) as f:
            data = yaml.safe_load(f)
        print(f'  - {os.path.basename(cfg_path)}: epochs={data.get("epochs")}, lr={data.get("lr")}')
    else:
        print(f'  [ERR] Không tìm thấy {cfg_path}')

print('\n[OK] Toàn bộ cấu hình STAIR-NLGCL v4 đã sẵn sàng!')

## Cell 4 — Helper Functions & Training Runner (Kèm VRAM Profiler & Loss Logger)


In [ ]:
# Cell 4: Hàm hỗ trợ chạy Training & Giám sát Phần cứng
import subprocess, threading, time, os, re, sys

vram_profile = {}

def vram_monitor(key, stop_evt, interval=2.0):
    try:
        import pynvml
        pynvml.nvmlInit()
        h = pynvml.nvmlDeviceGetHandleByIndex(0)
        records = []
        while not stop_evt.is_set():
            mem = pynvml.nvmlDeviceGetMemoryInfo(h)
            records.append(mem.used / 1024**2)
            time.sleep(interval)
        pynvml.nvmlShutdown()
        vram_profile[key] = records
    except Exception as e:
        vram_profile[key] = []

def extract_best_test(log_path):
    """Parse log file for best TEST metrics and best checkpoint epoch."""
    if not os.path.exists(log_path):
        return None, None
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
        lines = content.splitlines()
    
    best_epoch = None
    best_metrics = {}
    
    # 1. Tìm best epoch từ các mẫu log khác nhau của FreeRec
    ep_matches = re.findall(r'(?:Load best model @Epoch|TEST @Epoch:|Best @Epoch:?)\s*(\d+)', content, re.IGNORECASE)
    if ep_matches:
        best_epoch = int(ep_matches[-1])
    else:
        # Fallback: tìm từ các dòng cuối có nhắc đến epoch
        for line in reversed(lines):
            m = re.search(r'Epoch:\s*(\d+)', line)
            if m:
                best_epoch = int(m.group(1))
                break
    
    # 2. Tìm test metrics
    for line in lines:
        if any(k in line for k in ['Recall@20', 'NDCG@20', 'Recall@10', 'NDCG@10']):
            for metric in ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']:
                m = re.search(rf'{metric}\s*Avg:\s*([0-9.]+)', line, re.IGNORECASE)
                if m:
                    best_metrics[metric] = float(m.group(1))
    
    return best_epoch, best_metrics

def parse_training_loss(log_path):
    """Parse per-epoch training loss."""
    if not os.path.exists(log_path):
        return []
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    matches = re.findall(r'TRAIN @Epoch:\s*(\d+).*?LOSS\s+Avg:\s*([0-9.]+)', content, re.DOTALL)
    return [(int(ep), float(loss)) for ep, loss in matches]

def parse_valid_metrics(log_path):
    """Parse per-epoch validation NDCG@20."""
    if not os.path.exists(log_path):
        return []
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    matches = re.findall(r'VALID @Epoch:\s*(\d+).*?NDCG@20\s+Avg:\s*([0-9.]+)', content, re.DOTALL)
    return [(int(ep), float(v)) for ep, v in matches]

def run_training_v4(key, yaml_cfg, data_root, log_path,
                    lambda_nlgcl=0.01, nlgcl_tau=0.2, nlgcl_G=1, nlgcl_alpha=0.5):
    print('=' * 60)
    print(f'BẮT ĐẦU HUẤN LUYỆN v4 (STAIR-NLGCL): {key.upper()}')
    print(f'Config     : {yaml_cfg}')
    print(f'Log        : {log_path}')
    print(f'λ_nlgcl    : {lambda_nlgcl}')
    print(f'τ (tau)    : {nlgcl_tau}')
    print(f'G (gaps)   : {nlgcl_G}')
    print(f'α (alpha)  : {nlgcl_alpha}')
    print('=' * 60)
    
    stop_evt = threading.Event()
    th = threading.Thread(target=vram_monitor, args=(key, stop_evt), daemon=True)
    th.start()
    
    t0 = time.time()
    cmd = [
        sys.executable, '/kaggle/working/STAIR-Enhanced/main_stair_nlgcl_v4.py',
        '--config', yaml_cfg,
        '--root',   data_root,
        '--lambda-nlgcl',  str(lambda_nlgcl),
        '--nlgcl-tau',     str(nlgcl_tau),
        '--nlgcl-G',       str(nlgcl_G),
        '--nlgcl-alpha',   str(nlgcl_alpha),
    ]
    with open(log_path, 'w', encoding='utf-8') as f:
        result = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT,
                                cwd='/kaggle/working/STAIR-Enhanced')
    
    elapsed = time.time() - t0
    stop_evt.set()
    th.join(timeout=3)
    
    if result.returncode != 0:
        print(f'[THẤT BẠI] Mã lỗi {result.returncode} (Thời gian: {elapsed/60:.1f} phút)')
        with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
            print('\n'.join(f.readlines()[-30:]))
    else:
        print(f'[HOÀN THÀNH] {key.upper()} trong {elapsed/60:.1f} phút')
        ep, metrics = extract_best_test(log_path)
        if metrics:
            print(f'  - Best checkpoint @Epoch: {ep}')
            for k, v in metrics.items():
                print(f'    * {k}: {v:.6f}')
        if key in vram_profile and vram_profile[key]:
            peak = max(vram_profile[key])
            print(f'  - VRAM Peak: {peak:.0f} MB')
    return result.returncode

print('[OK] Helper functions defined with robust epoch parsing.')

## Cell 5 — Cấu hình Siêu tham số v4 (NLGCL+ Multimodal Scale)

### 🔬 Cơ sở Lý thuyết Thang đo Trọng số λ_nlgcl (NLGCL vs NLGCL+)
- **NLGCL gốc (WWW 2024, LightGCN đơn phương thái):** Tìm kiếm λ trong thang `{1e-6, 1e-5, 1e-4}`.
- **NLGCL+ (Multimodal GCL trên FREEDOM / MMRec):** Tìm kiếm λ trong thang **`{1e-3, 1e-2, 1e-1}`** và chứng minh **`λ = 1e-2 (0.01)`** đạt tối ưu nhất quán trên mọi tập dữ liệu đa phương thức.

### 📊 Grid Search Plan (NLGCL+ Multimodal Scale)
| Lần chạy | λ_nlgcl | τ | G | Mục tiêu & Cơ sở lý thuyết |
|:---:|:---:|:---:|:---:|:---|
| **Run 1 (Ưu tiên)** | `1e-2` (0.01) | `0.2` | `1` | **Giá trị tối ưu chuẩn theo NLGCL+** (khuyến nghị cho multimodal) |
| **Run 2** | `1e-3` (0.001) | `0.2` | `1` | Tín hiệu CL nhẹ hơn 10x nếu 1e-2 làm chệch BPR |
| **Run 3** | `1e-1` (0.1) | `0.2` | `1` | Tín hiệu CL mạnh hơn 10x nếu 1e-2 vẫn chưa đủ mạnh |


In [ ]:
# Cell 5: Cấu hình Siêu tham số STAIR-NLGCL v4
# ══════════════════════════════════════════════════════════════
# TẤT CẢ tham số STAIR Baseline giữ nguyên (kế thừa từ YAML configs)
# CHỈ thiết lập các tham số NLGCL mới:
# ══════════════════════════════════════════════════════════════

LAMBDA_NLGCL = 1e-2    # 0.01 (Chuẩn tối ưu theo NLGCL+ Multimodal)    # Trọng số NLGCL loss (bắt đầu yếu, tăng dần)
NLGCL_TAU    = 0.2     # Nhiệt độ InfoNCE (τ)
NLGCL_G      = 1       # Số khoảng cách tầng đối chiếu (G=1: Layer 0↔1)
NLGCL_ALPHA  = 0.5     # Cân bằng: α·L_user + (1-α)·L_item

# Paths
STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
DATA_ROOT = '/kaggle/data'
LOG_DIR   = '/kaggle/working/logs_nlgcl_v4'

import os
os.makedirs(LOG_DIR, exist_ok=True)

print('═' * 60)
print('STAIR-NLGCL v4 — CẤU HÌNH SIÊU THAM SỐ')
print('═' * 60)
print(f'  λ_nlgcl     : {LAMBDA_NLGCL}')
print(f'  τ (tau)      : {NLGCL_TAU}')
print(f'  G (gaps)     : {NLGCL_G}')
print(f'  α (alpha)    : {NLGCL_ALPHA}')
print(f'  STAIR_DIR    : {STAIR_DIR}')
print(f'  DATA_ROOT    : {DATA_ROOT}')
print(f'  LOG_DIR      : {LOG_DIR}')
print('═' * 60)
print()
print('Các tham số STAIR Baseline (kế thừa từ YAML, KHÔNG thay đổi):')
print('  optimizer=adamwsevo, lr=1e-3, epochs=500, batch_size=1024')
print('  embedding_dim=64, num_layers=3, gamma=0.1~0.2')


## Cell 6 — Huấn luyện STAIR-NLGCL v4 trên Baby & Sports


In [ ]:
# Cell 6: Huấn luyện STAIR-NLGCL v4 trên Baby & Sports
import torch, os

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
DATA_ROOT = '/kaggle/data'
LOG_DIR   = '/kaggle/working/logs_nlgcl_v4'
os.makedirs(LOG_DIR, exist_ok=True)

# 1. Huấn luyện Baby
run_training_v4(
    key          = 'baby',
    yaml_cfg     = f'{STAIR_DIR}/configs/Amazon2014Baby_550_MMRec.yaml',
    data_root    = DATA_ROOT,
    log_path     = f'{LOG_DIR}/baby.log',
    lambda_nlgcl = LAMBDA_NLGCL,
    nlgcl_tau    = NLGCL_TAU,
    nlgcl_G      = NLGCL_G,
    nlgcl_alpha  = NLGCL_ALPHA,
)
torch.cuda.empty_cache()

# 2. Huấn luyện Sports
run_training_v4(
    key          = 'sports',
    yaml_cfg     = f'{STAIR_DIR}/configs/Amazon2014Sports_550_MMRec.yaml',
    data_root    = DATA_ROOT,
    log_path     = f'{LOG_DIR}/sports.log',
    lambda_nlgcl = LAMBDA_NLGCL,
    nlgcl_tau    = NLGCL_TAU,
    nlgcl_G      = NLGCL_G,
    nlgcl_alpha  = NLGCL_ALPHA,
)
torch.cuda.empty_cache()

print('=' * 60)
print('✅ HOÀN THÀNH HUẤN LUYỆN BABY & SPORTS (STAIR-NLGCL v4)!')
print('=' * 60)

## Cell 7 — Huấn luyện STAIR-NLGCL v4 trên Electronics (Tập lớn nhất ~1.7M tương tác)


In [ ]:
# Cell 7: Huấn luyện STAIR-NLGCL v4 trên Electronics
import torch, os

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
DATA_ROOT = '/kaggle/data'
LOG_DIR   = '/kaggle/working/logs_nlgcl_v4'
os.makedirs(LOG_DIR, exist_ok=True)

run_training_v4(
    key          = 'electronics',
    yaml_cfg     = f'{STAIR_DIR}/configs/Amazon2014Electronics_550_MMRec.yaml',
    data_root    = DATA_ROOT,
    log_path     = f'{LOG_DIR}/electronics.log',
    lambda_nlgcl = LAMBDA_NLGCL,
    nlgcl_tau    = NLGCL_TAU,
    nlgcl_G      = NLGCL_G,
    nlgcl_alpha  = NLGCL_ALPHA,
)
torch.cuda.empty_cache()

print('=' * 60)
print('✅ HOÀN THÀNH HUẤN LUYỆN ELECTRONICS (STAIR-NLGCL v4)!')
print('=' * 60)

## Cell 8 — Tổng hợp Kết quả & So sánh Ablation Study 5 Phiên bản


In [ ]:
# Cell 8: Bảng so sánh Ablation Study: Baseline vs v1 vs v2a vs v3 vs v4
from prettytable import PrettyTable
import os, math

LOG_DIR_V4 = '/kaggle/working/logs_nlgcl_v4'

BASELINE = {
    'baby':        {'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
    'sports':      {'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
    'electronics': {'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
}

V1_RESULTS = {
    'baby':        {'Recall@10': 0.0611, 'Recall@20': 0.0948, 'NDCG@10': 0.0325, 'NDCG@20': 0.0412},
    'sports':      {'Recall@10': 0.0695, 'Recall@20': 0.1040, 'NDCG@10': 0.0376, 'NDCG@20': 0.0466},
    'electronics': {'Recall@10': 0.0401, 'Recall@20': 0.0601, 'NDCG@10': 0.0223, 'NDCG@20': 0.0274},
}

V2A_RESULTS = {
    'baby':        {'Recall@10': 0.0663, 'Recall@20': 0.1026, 'NDCG@10': 0.0351, 'NDCG@20': 0.0445},
    'sports':      {'Recall@10': 0.0738, 'Recall@20': 0.1102, 'NDCG@10': 0.0401, 'NDCG@20': 0.0494},
    'electronics': {'Recall@10': 0.0435, 'Recall@20': 0.0658, 'NDCG@10': 0.0241, 'NDCG@20': 0.0298},
}

V3_RESULTS = {
    'baby':        {'Recall@10': 0.0680, 'Recall@20': 0.1050, 'NDCG@10': 0.0362, 'NDCG@20': 0.0458},
    'sports':      {'Recall@10': 0.0750, 'Recall@20': 0.1120, 'NDCG@10': 0.0410, 'NDCG@20': 0.0506},
    'electronics': {'Recall@10': 0.0445, 'Recall@20': 0.0670, 'NDCG@10': 0.0248, 'NDCG@20': 0.0306},
}

v4_results = {}
for ds in ['baby', 'sports', 'electronics']:
    log = os.path.join(LOG_DIR_V4, f'{ds}.log')
    ep, metrics = extract_best_test(log)
    v4_results[ds] = {'epoch': ep, 'metrics': metrics}

METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

print('=' * 130)
print(f'BẢNG SO SÁNH ABLATION STUDY: Baseline vs v1 vs v2a vs v3 (LIA) vs v4 (NLGCL, λ={LAMBDA_NLGCL})')
print('=' * 130)

for ds in ['baby', 'sports', 'electronics']:
    t = PrettyTable()
    t.field_names = ['Chỉ số', 'STAIR BL', 'v1 (Replace)', 'v2a (Dynamic)', 'v3 (LIA)', 'v4 (NLGCL)', 'Δ(v4 vs BL)']
    t.align = 'r'; t.align['Chỉ số'] = 'l'
    bl  = BASELINE[ds]
    v1  = V1_RESULTS[ds]
    v2a = V2A_RESULTS[ds]
    v3  = V3_RESULTS[ds]
    v4  = v4_results[ds]['metrics'] or {}
    for m in METRICS:
        bl_val  = bl.get(m, float('nan'))
        v1_val  = v1.get(m, float('nan'))
        v2a_val = v2a.get(m, float('nan'))
        v3_val  = v3.get(m, float('nan'))
        v4_val  = v4.get(m, float('nan'))
        delta = f'{(v4_val - bl_val)/bl_val*100:+.2f}%' if (bl_val and v4_val and not math.isnan(v4_val)) else 'N/A'
        t.add_row([m, f'{bl_val:.4f}', f'{v1_val:.4f}', f'{v2a_val:.4f}', f'{v3_val:.4f}',
                   f'{v4_val:.4f}' if (v4_val and not math.isnan(v4_val)) else 'N/A', delta])
    ep_str = v4_results[ds]['epoch']
    print(f'\nTập dữ liệu: {ds.upper()} (Best Epoch v4: {ep_str})')
    print(t)
print('=' * 130)

## Cell 9 — Biểu đồ Learning Curves & VRAM Profiling


In [ ]:
# Cell 9: Vẽ Learning Curves (Training Loss + Validation NDCG@20)
import matplotlib.pyplot as plt
import os

LOG_DIR = '/kaggle/working/logs_nlgcl_v4'

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
fig.suptitle(f'STAIR-NLGCL v4: Learning Curves (λ={LAMBDA_NLGCL}, τ={NLGCL_TAU}, G={NLGCL_G})',
             fontsize=14, fontweight='bold')

# Row 1: Training Loss
for i, ds in enumerate(['baby', 'sports', 'electronics']):
    ax = axes[0][i]
    log_path = os.path.join(LOG_DIR, f'{ds}.log')
    loss_history = parse_training_loss(log_path)
    if loss_history:
        epochs = [h[0] for h in loss_history]
        losses = [h[1] for h in loss_history]
        ax.plot(epochs, losses, 'b-', linewidth=1.5, alpha=0.8, label='Total Loss')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Training Loss')
        ax.set_title(f'{ds.capitalize()} — Training Loss')
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
        min_idx = losses.index(min(losses))
        ax.axvline(x=epochs[min_idx], color='r', linestyle=':', alpha=0.5)
    else:
        ax.text(0.5, 0.5, 'Chưa có log', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{ds.capitalize()} (No data)')

# Row 2: Validation NDCG@20
for i, ds in enumerate(['baby', 'sports', 'electronics']):
    ax = axes[1][i]
    log_path = os.path.join(LOG_DIR, f'{ds}.log')
    valid_history = parse_valid_metrics(log_path)
    if valid_history:
        epochs = [h[0] for h in valid_history]
        ndcgs  = [h[1] for h in valid_history]
        ax.plot(epochs, ndcgs, 'g-', linewidth=1.5, alpha=0.8, label='Valid NDCG@20')
        bl_ndcg = BASELINE.get(ds, {}).get('NDCG@20', 0)
        if bl_ndcg:
            ax.axhline(y=bl_ndcg, color='red', linestyle='--', linewidth=1.5,
                      label=f'Baseline ({bl_ndcg:.4f})')
        max_idx = ndcgs.index(max(ndcgs))
        ax.axvline(x=epochs[max_idx], color='purple', linestyle=':', alpha=0.5)
        ax.annotate(f'Best: {ndcgs[max_idx]:.4f}\n@Ep {epochs[max_idx]}',
                   xy=(epochs[max_idx], ndcgs[max_idx]),
                   fontsize=8, color='purple',
                   xytext=(10, -20), textcoords='offset points',
                   arrowprops=dict(arrowstyle='->', color='purple', alpha=0.5))
        ax.set_xlabel('Epoch'); ax.set_ylabel('NDCG@20')
        ax.set_title(f'{ds.capitalize()} — Valid NDCG@20')
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'Chưa có log', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{ds.capitalize()} (No data)')

plt.tight_layout()
plt.savefig('/kaggle/working/learning_curves_v4_nlgcl.png', dpi=150, bbox_inches='tight')
plt.show()
print('[ĐÃ LƯU] /kaggle/working/learning_curves_v4_nlgcl.png')

## Cell 10 — Biểu đồ VRAM Usage


In [ ]:
# Cell 10: Vẽ VRAM Profiling
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('STAIR-NLGCL v4: VRAM Usage Over Training', fontsize=14, fontweight='bold')

for i, ds in enumerate(['baby', 'sports', 'electronics']):
    ax = axes[i]
    if ds in vram_profile and vram_profile[ds]:
        data = vram_profile[ds]
        ax.plot(range(len(data)), data, 'b-', linewidth=1, alpha=0.7)
        ax.axhline(y=max(data), color='r', linestyle='--', linewidth=1.5,
                  label=f'Peak: {max(data):.0f} MB')
        ax.axhline(y=sum(data)/len(data), color='g', linestyle=':', linewidth=1,
                  label=f'Avg: {sum(data)/len(data):.0f} MB')
        ax.set_xlabel('Sample (every 2s)'); ax.set_ylabel('VRAM (MB)')
        ax.set_title(f'{ds.capitalize()}')
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'Chưa có dữ liệu', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'{ds.capitalize()} (No data)')

plt.tight_layout()
plt.savefig('/kaggle/working/vram_profile_v4_nlgcl.png', dpi=150, bbox_inches='tight')
plt.show()
print('[ĐÃ LƯU] /kaggle/working/vram_profile_v4_nlgcl.png')

## Cell 11 — Xuất Bảng Kết quả CSV cho Khóa luận Tốt nghiệp


In [ ]:
# Cell 11: Xuất bảng kết quả CSV
import csv, os

OUT_CSV = '/kaggle/working/ablation_enhanced_v4_nlgcl.csv'
METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

rows = []
for ds in ['baby', 'sports', 'electronics']:
    bl  = BASELINE[ds]
    v1  = V1_RESULTS[ds]
    v2a = V2A_RESULTS[ds]
    v3  = V3_RESULTS[ds]
    v4  = v4_results[ds]['metrics'] or {}
    ep  = v4_results[ds]['epoch']
    for m in METRICS:
        bl_v  = bl.get(m)
        v1_v  = v1.get(m)
        v2_v  = v2a.get(m)
        v3_v  = v3.get(m)
        v4_v  = v4.get(m)
        delta_v4 = (v4_v - bl_v)/bl_v*100 if (bl_v and v4_v) else None
        rows.append({'dataset': ds.capitalize(), 'metric': m, 'model': 'STAIR-Baseline', 'value': f'{bl_v:.4f}' if bl_v else '', 'best_epoch': ''})
        rows.append({'dataset': ds.capitalize(), 'metric': m, 'model': 'Enhanced-v1', 'value': f'{v1_v:.4f}' if v1_v else '', 'best_epoch': ''})
        rows.append({'dataset': ds.capitalize(), 'metric': m, 'model': 'Enhanced-v2a', 'value': f'{v2_v:.4f}' if v2_v else '', 'best_epoch': ''})
        rows.append({'dataset': ds.capitalize(), 'metric': m, 'model': 'Enhanced-v3 (LIA)', 'value': f'{v3_v:.4f}' if v3_v else '', 'best_epoch': ''})
        rows.append({'dataset': ds.capitalize(), 'metric': m, 'model': f'Enhanced-v4 (NLGCL)', 'value': f'{v4_v:.4f}' if v4_v else 'N/A', 'best_epoch': str(ep) if ep else 'N/A'})
        if delta_v4 is not None:
            rows.append({'dataset': ds.capitalize(), 'metric': m, 'model': 'Delta v4 vs BL (%)', 'value': f'{delta_v4:+.2f}%', 'best_epoch': ''})

with open(OUT_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['dataset', 'metric', 'model', 'value', 'best_epoch'])
    writer.writeheader()
    writer.writerows(rows)

print(f'[ĐÃ XUẤT CSV] {OUT_CSV}')

## 📋 Hướng dẫn Tinh chỉnh Grid Search cho λ_nlgcl (Theo thang NLGCL+)

### Quy trình Thực nghiệm 3 bước:

**Bước 1:** Chạy cấu hình chuẩn với `LAMBDA_NLGCL = 1e-2` (0.01).
- Đây là giá trị được NLGCL+ chứng minh tối ưu nhất quán trên các mô hình gợi ý đa phương thức.
- Theo dõi sự thay đổi của Best Epoch và NDCG@20 trên cả Baby và Sports.

**Bước 2:** Điều chỉnh biến `LAMBDA_NLGCL` ở Cell 5 nếu cần mở rộng biên độ:
```python
# Nếu 1e-2 cho kết quả tốt, thử đẩy mạnh hơn:
LAMBDA_NLGCL = 1e-1    # 0.1

# Nếu 1e-2 làm BPR loss tăng >20% hoặc early-stopping quá sớm:
LAMBDA_NLGCL = 1e-3    # 0.001
```

**Bước 3:** Quan sát bảng Ablation Study 5 phiên bản tại Cell 8.

### ⚠️ Phân tích Hiện tượng Best-Epoch Shift:
- Nếu tăng λ giúp NDCG@20 tăng $\rightarrow$ Tín hiệu tương phản đang hướng dẫn mô hình học biểu diễn tốt hơn.
- Nếu tăng λ làm Best Epoch dịch chuyển quá sớm mà NDCG@20 giảm $\rightarrow$ Nhiễu tương phản gây kích hoạt early-stopping trước khi hội tụ BPR.